# 02 — Clean & Merge Stadium Data

Takes the raw pulls from `01_stadiums_pull.ipynb` and:

1. Cleans and merges the two Wikipedia stadium tables with the ASA stadia table.
2. Identifies each team's home stadium per year (most games played that year at that stadium).
3. Derives team lifespans (expansion year / fold year) from active seasons.
4. Ranks stadium tenants (primary vs. secondary) and assigns each stadium a `competition` label (`nwsl`, `mls`, or `both`).
5. Fills in missing stadium coordinates (geocoding) and missing capacities (manual lookup for a handful of new/renamed venues).

**Inputs:** `data/raw/games_raw.csv`, `players_raw.csv`, `teams_raw.csv`, `stadia_raw.csv`, `stadiums_wiki_nwsl_raw.csv`, `stadiums_wiki_mls_raw.csv`
**Outputs:** `data/processed/stadiums.csv`, `data/processed/teams_clean.csv`, `data/processed/games_clean.csv`


In [ ]:
import pandas as pd
import numpy as np
import os
import time
from geopy.geocoders import Nominatim

DATA_RAW_DIR = os.path.join("..", "data", "raw")
DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

games = pd.read_csv(os.path.join(DATA_RAW_DIR, "games_raw.csv"))
teams = pd.read_csv(os.path.join(DATA_RAW_DIR, "teams_raw.csv"))
stadia = pd.read_csv(os.path.join(DATA_RAW_DIR, "stadia_raw.csv"))
stadiums_wiki_nwsl = pd.read_csv(os.path.join(DATA_RAW_DIR, "stadiums_wiki_nwsl_raw.csv"))
stadiums_wiki_mls = pd.read_csv(os.path.join(DATA_RAW_DIR, "stadiums_wiki_mls_raw.csv"))

print(f"Loaded: games {games.shape}, teams {teams.shape}, stadia {stadia.shape}, "
      f"wiki_nwsl {stadiums_wiki_nwsl.shape}, wiki_mls {stadiums_wiki_mls.shape}")


## Merge Wikipedia stadium tables with ASA stadia

In [ ]:
# The MLS wiki table carries columns we don't need (image links, field
# dimensions, footnote refs) -- drop them before merging
stadiums_wiki_mls = stadiums_wiki_mls.drop(columns=["Image", "Field dimensions", "Ref(s)"], errors="ignore")

# Wikipedia table cells often have footnote markers like "Stadium Name[1]" --
# strip everything from the first "[" onward across all text columns
def strip_footnotes(df):
    return df.apply(
        lambda col: col.astype(str).str.split("[").str[0].str.strip() if col.dtype == "object" else col
    )

stadiums_wiki_nwsl = strip_footnotes(stadiums_wiki_nwsl)
stadiums_wiki_mls = strip_footnotes(stadiums_wiki_mls)

stad = stadiums_wiki_nwsl.merge(
    stadiums_wiki_mls, on="Stadium", how="outer", suffixes=["_nwsl", "_mls"]
)
print(f"Merged wiki stadium table: {stad.shape}")


In [ ]:
# Dedup the ASA stadia table, and flag stadiums shared by both leagues
stadiums = stadia.drop_duplicates(subset=["stadium_name", "stadium_id"]).copy()

dup_mask = stadia.duplicated(subset=["stadium_name", "stadium_id"], keep=False)
duplicated_names = stadia.loc[dup_mask, "stadium_name"]

stadiums["competition"] = np.where(
    stadiums["stadium_name"].isin(duplicated_names),
    "both",
    stadiums["competition"],
)

# Bring in the Wikipedia metadata
stadiums_merged = stadiums.merge(stad, left_on="stadium_name", right_on="Stadium", how="left")
print(f"ASA + wiki merged stadium table: {stadiums_merged.shape}")


## Identify each team's home stadium, by year

A team can play at more than one venue in a season (temporary relocations, doubleheaders, etc.), so "home stadium" is defined as whichever stadium hosted the most of that team's home games in a given year. A `NaN` home stadium for a given (year, team) pair means the team didn't play that year -- useful for spotting expansion/folding years.


In [ ]:
games["date_time_utc"] = pd.to_datetime(games["date_time_utc"])
games["year"] = games["date_time_utc"].dt.year

stadium_counts = (
    games.groupby(["year", "home_team_id", "stadium_id"])
    .size()
    .reset_index(name="games_played")
)

home_stadiums = (
    stadium_counts.sort_values("games_played", ascending=False)
    .drop_duplicates(subset=["year", "home_team_id"])
    [["year", "home_team_id", "stadium_id"]]
    .rename(columns={"stadium_id": "home_stadium"})
)

# Reindex to every (year, team) combination so missing years show up as NaN
full_index = pd.MultiIndex.from_product(
    [games["year"].unique(), games["home_team_id"].unique()],
    names=["year", "home_team_id"],
)

home_stadium_by_year = (
    home_stadiums.set_index(["year", "home_team_id"])
    .reindex(full_index)
    .reset_index()
    .sort_values(["home_team_id", "year"])
)

print(f"home_stadium_by_year: {home_stadium_by_year.shape}, "
      f"{home_stadium_by_year['home_stadium'].isna().sum()} team-years with no home stadium")


## Derive team lifespans (expansion year / fold year)

In [ ]:
active_seasons = home_stadium_by_year.dropna(subset=["home_stadium"])

team_lifespans = (
    active_seasons.groupby("home_team_id")["year"]
    .agg(expansion_year="min", last_active_year="max")
    .reset_index()
)

max_dataset_year = games["year"].max()
team_lifespans["year_of_death"] = team_lifespans["last_active_year"].apply(
    lambda y: y + 1 if y < max_dataset_year else pd.NA
)

# Join lifespan info onto teams (drop any stale columns from a prior run first)
cols_to_drop = [c for c in teams.columns if "expansion_year" in c or "year_of_death" in c]
teams = teams.drop(columns=cols_to_drop)

teams = teams.merge(
    team_lifespans[["home_team_id", "expansion_year", "year_of_death"]],
    left_on="team_id", right_on="home_team_id", how="left",
).drop(columns=["home_team_id"])

print(f"teams with lifespan info: {teams.shape}")
teams.to_csv(os.path.join(DATA_PROCESSED_DIR, "teams_clean.csv"), index=False)
games.to_csv(os.path.join(DATA_PROCESSED_DIR, "games_clean.csv"), index=False)
print("Saved teams_clean.csv, games_clean.csv")


## Rank stadium tenants and assign a final `competition` label

For each stadium, rank the teams that called it home by total years played there (rank 1 = primary tenant, rank 2 = secondary tenant). This lets us assign each stadium a `competition` value (`nwsl`, `mls`, or `both`) based on who actually plays there, rather than relying only on the ASA/Wikipedia source tables.


In [ ]:
stadium_tenures = home_stadium_by_year.dropna(subset=["home_stadium"]).copy()

stadium_summary = (
    stadium_tenures.groupby(["home_stadium", "home_team_id"])
    .agg(
        start_year=("year", "min"),
        end_year=("year", "max"),
        total_years=("year", "nunique"),
    )
    .reset_index()
)

stadium_summary["years_played"] = stadium_summary.apply(
    lambda row: f"{row['start_year']}-{row['end_year']}" if row["start_year"] != row["end_year"] else str(row["start_year"]),
    axis=1,
)

latest_year = games["year"].max()
stadium_summary["is_current"] = stadium_summary["end_year"] == latest_year

stadium_summary["tenant_rank"] = (
    stadium_summary.groupby("home_stadium")["total_years"]
    .rank(method="first", ascending=False)
)

primary_tenants = stadium_summary[stadium_summary["tenant_rank"] == 1].set_index("home_stadium")
secondary_tenants = stadium_summary[stadium_summary["tenant_rank"] == 2].set_index("home_stadium")

stadium_ids = stadiums_merged["stadium_id"]

stadiums_merged["primary_team_id"] = stadium_ids.map(primary_tenants["home_team_id"])
stadiums_merged["primary_is_current"] = stadium_ids.map(primary_tenants["is_current"]).fillna(False)
stadiums_merged["primary_years_played"] = stadium_ids.map(primary_tenants["years_played"])

stadiums_merged["secondary_team_id"] = stadium_ids.map(secondary_tenants["home_team_id"])
stadiums_merged["secondary_is_current"] = stadium_ids.map(secondary_tenants["is_current"]).fillna(False)
stadiums_merged["secondary_years_played"] = stadium_ids.map(secondary_tenants["years_played"])


def get_league(team_id):
    if pd.isna(team_id):
        return None
    match = teams.loc[teams["team_id"] == team_id, "league"]
    return match.iloc[0] if not match.empty else None


def set_competition(row):
    has_primary = pd.notna(row["primary_team_id"])
    has_secondary = pd.notna(row["secondary_team_id"])
    if has_primary and has_secondary:
        return "both"
    elif has_primary:
        return get_league(row["primary_team_id"])
    elif has_secondary:
        return get_league(row["secondary_team_id"])
    else:
        return None


stadiums_merged["competition"] = stadiums_merged.apply(set_competition, axis=1)
print(f"stadiums_merged with tenant info: {stadiums_merged.shape}")


## Final stadium list: fill missing coordinates and capacities

Restrict to stadiums that actually have a primary tenant, then:
- Geocode any stadium still missing lat/lon via Nominatim (OpenStreetMap).
- Fill remaining missing capacities using a manual lookup for a handful of new/renamed venues not yet in the source data.


In [ ]:
final_stadiums = stadiums_merged.dropna(subset=["primary_team_id"]).copy()
n_missing_coords = final_stadiums["latitude"].isna().sum()
print(f"{n_missing_coords} stadiums missing coordinates before geocoding")

geolocator = Nominatim(user_agent="stadium_geocoder")


def geocode_stadium(row):
    # Skip rows that already have coordinates
    if pd.notna(row["latitude"]) and pd.notna(row["longitude"]):
        return row["latitude"], row["longitude"]

    query = f"{row['stadium_name']}, {row['city'] if pd.notna(row['city']) else ''}"
    try:
        time.sleep(1)  # Respect Nominatim's rate limit
        location = geolocator.geocode(query)
        if location:
            return location.latitude, location.longitude
    except Exception:
        pass
    return None, None


coords = final_stadiums.apply(geocode_stadium, axis=1)
final_stadiums["latitude"] = [c[0] for c in coords]
final_stadiums["longitude"] = [c[1] for c in coords]

# Manual capacity lookup for stadiums missing capacity in both source tables
# (mostly new/temporary/renamed venues as of the last data pull)
capacity_map = {
    "18KXMe8lXQ64": 12000,  # Northwestern (Martin Stadium / temporary lakeside)
    "25Oa5wKXY514": 35000,  # Snapdragon Stadium
    "37gpMOrLOQzy": 10000,  # WakeMed Soccer Park
    "47xW5p3L0Mg1": 11500,  # CPKC Stadium
    "78odMXgYWQYL": 26700,  # Miami Freedom Park
}
final_stadiums["capacity"] = final_stadiums["capacity"].fillna(
    final_stadiums["stadium_id"].map(capacity_map)
)

final_stadiums = final_stadiums.reset_index(drop=True)
print(f"Final stadium table: {final_stadiums.shape}, "
      f"{final_stadiums['latitude'].isna().sum()} still missing coordinates, "
      f"{final_stadiums['capacity'].isna().sum()} still missing capacity")

final_stadiums.to_csv(os.path.join(DATA_PROCESSED_DIR, "stadiums.csv"), index=False)
print("Saved stadiums.csv")
